# Memprediksi Gagal Bayar Pinjaman UMKM: Sinyal Terkuat Ada di Rasio Pengembalian

| | |
| --- | --- |
| **Kompetisi** | [ADIKARA 2025: Indonesian Credit Score](https://www.kaggle.com/competitions/adikara-2025-indonesia-kredit-macet) (Fakultas Informatika Telkom University, Kaggle, Desember 2025) |
| **Tujuan** | Memprediksi `status_gagal_bayar` (1 = gagal bayar) untuk transaksi pinjaman digital (metrik macro F1) |
| **Data** | 20.012 transaksi berlabel dan 5.003 transaksi uji, 11 atribut pinjaman dan peminjam |
| **Penulis** | Fauzan Ahsanudin Alfikri · [Kaggle: fauzan2](https://www.kaggle.com/fauzan2) |
| **Tanggal** | Disusun ulang November 2026, menggantikan notebook kompetisi Desember 2025 |
| **Lingkungan** | Dijalankan penuh di Kaggle (CPU) |

**Pertanyaan penelitian.** Atribut apa yang paling membedakan pinjaman yang gagal bayar, dan bagaimana mengubah
probabilitas model menjadi label 0/1 yang memaksimalkan macro F1?

**Ringkasan temuan.**
- **Rasio pengembalian** (total pengembalian dibagi pokok pinjaman) adalah sinyal paling kuat: AUC 0,984 sendirian. Pinjaman dengan rasio ≤ 1,01 hampir tidak pernah gagal bayar, sedangkan rasio di atas 1,2 gagal bayar 92%.
- Atribut peminjam seperti provinsi, sektor usaha, pendidikan, dan jaminan hampir tidak berpengaruh (4,7–5,5% gagal bayar di semua kategori). Yang berbeda hanya peminjam baru (41%) dan jenis pinjaman di luar modal usaha.
- Data kosong di train (24% baris) jauh lebih banyak daripada di test (7%). Validasi utama dihitung pada baris yang total pengembaliannya terisi, karena itulah kondisi 99% baris test.
- LightGBM dengan rasio pengembalian dan waktu pencairan mencapai **macro F1 OOF 0,947** (atribut mentah saja 0,851). Late submission: public **0,9563**, private **0,9450**; submission tim saat kompetisi mencapai private tertinggi 0,9471 (peringkat akhir 6 dari 49).
- Tanpa total pengembalian dan porsi lender (atribut yang kemungkinan baru diketahui setelah pinjaman berjalan), macro F1 turun ke 0,667.

**Alur:** 1. Setup → 2. Memahami data → 3. EDA → 4. Preprocessing dan desain validasi → 5. Baseline →
6. Feature engineering dan ablation → 7. Ambang keputusan → 8. Interpretasi → 9. Submission → 10. Kesimpulan

## 1. Setup

Semua import, parameter, dan path ada di sel ini. Fitur turunan ada di `src/features`, dan cross-validation serta pemilihan
ambang ada di `src/models`.

In [ ]:
import os
import shutil
import sys
import time
import warnings
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SRC_ROOT = Path.cwd().parent
for r, _, f in os.walk("/kaggle/input", followlinks=True):
    if Path(r).name == "features" and "build_features.py" in f:
        pkg = Path(r).parent
        if pkg.name == "src":
            SRC_ROOT = pkg.parent
        else:
            SRC_ROOT = Path("/tmp/project")
            shutil.copytree(pkg, SRC_ROOT / "src", dirs_exist_ok=True)
        break
sys.path.insert(0, str(SRC_ROOT))
from src.data.load_data import CATEGORICAL, ID, NUMERIC, TARGET, find_data_dir, load
from src.features.build_features import GROUPS, base_frame, build, derived, typical_lender_share
from src.models.train_model import THRESHOLDS, best_threshold, cv_predict, f1_curve, lgb_model, macro_f1_by_seed
from src.visualization.plots import PALETTE, pct_axis, save_fig, set_style

# ── Parameter eksperimen
SEEDS = range(5)                 # Stratified 5-Fold diulang dengan 5 seed

# ── Path
DATA_DIR = find_data_dir(local="../data/raw")
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../reports")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")
set_style()
C1, C2, C3, CN = PALETTE["normal"], PALETTE["fraud"], PALETTE["accent"], PALETTE["neutral"]
T0 = time.time()
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)

## 2. Memahami data

Setiap baris adalah satu pinjaman: tanggal pencairan, provinsi, jenis pinjaman, status peminjam (baru atau berulang),
jumlah pinjaman, total pengembalian, durasi, porsi pengembalian untuk lender, sektor usaha, pendidikan, dan jenis jaminan.

In [ ]:
train, test, sample = load(DATA_DIR)
y = train[TARGET].values
print(f"Train: {train.shape} | Test: {test.shape} | gagal bayar: {y.mean():.2%} ({y.sum():,} transaksi)")
miss = pd.DataFrame({"kosong train": train.isna().mean(), "kosong test": test.isna().mean()})
display(train.head(3))
miss[miss.sum(1) > 0].style.format("{:.1%}")

**Temuan.** 20.012 transaksi latih dengan 5,03% gagal bayar (1.006 transaksi), jadi kelasnya sangat tidak seimbang. Empat kolom punya nilai kosong. Yang penting: total pengembalian dan porsi lender kosong di sekitar 10% baris train, tetapi hanya sekitar 1% baris test.

## 3. Exploratory Data Analysis

### 3.1 Atribut kategori: hanya dua yang berbeda

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for a, c in zip(axes.flat, CATEGORICAL):
    g = train.groupby(train[c].fillna("(kosong)"))[TARGET].agg(["mean", "size"]).sort_values("mean")
    a.barh(g.index.astype(str), g["mean"], color=[C2 if v > 2 * y.mean() else C1 for v in g["mean"]])
    a.axvline(y.mean(), color=CN, ls="--", lw=1)
    for i, (m, n) in enumerate(zip(g["mean"], g["size"])):
        a.text(m, i, f" {m:.0%} (n={n:,})", va="center", fontsize=8)
    pct_axis(a, 0, axis="x"); a.set(title=c, xlabel="proporsi gagal bayar")
save_fig(FIG_DIR, "01_categorical")

**Temuan.** Provinsi, sektor usaha, pendidikan, dan jenis jaminan semuanya berada di sekitar rata-rata 5%, jadi atribut profil peminjam ini hampir tidak membawa informasi. Dua atribut yang berbeda: **peminjam baru** gagal bayar 41% (tetapi hanya 221 transaksi), dan jenis pinjaman di luar modal usaha (investasi 19%, mikro UKM 21%, multiguna 29%, konsumtif 83%). Modal usaha mendominasi data dan hanya 3–4% gagal bayar.

### 3.2 Rasio pengembalian: total pengembalian dibagi jumlah pinjaman

Pinjaman yang lancar dikembalikan hampir sebesar pokoknya. Grafik berikut melihat apa yang terjadi jika total
pengembalian jauh di atas pokok.

In [ ]:
markup = train.total_pengembalian / train.jumlah_pinjaman.replace(0, np.nan)
bins = [0, 1, 1.01, 1.03, 1.05, 1.1, 1.2, 1.5, np.inf]
mb = train.groupby(pd.cut(markup, bins, right=True, include_lowest=True), observed=True)[TARGET].agg(["mean", "size"])
fig, ax = plt.subplots(1, 2, figsize=(16, 4.6))
ax[0].bar([str(i) for i in mb.index], mb["mean"], color=C2)
for i, (m, n) in enumerate(zip(mb["mean"], mb["size"])):
    ax[0].text(i, m + 0.01, f"{m:.0%}\nn={n:,}", ha="center", fontsize=8)
pct_axis(ax[0], 0); ax[0].set(xlabel="rasio pengembalian (total / pokok)", ylabel="proporsi gagal bayar", title="Gagal bayar per rentang rasio pengembalian")
ax[0].tick_params(axis="x", rotation=30)
for k, col, lab in [(0, C1, "lancar"), (1, C2, "gagal bayar")]:
    ax[1].hist(markup[(y == k)].clip(0.95, 2), bins=60, alpha=0.6, density=True, color=col, label=lab)
ax[1].set(yscale="log", xlabel="rasio pengembalian (dipotong di 2)", ylabel="densitas (log)", title="Distribusi rasio per kelas"); ax[1].legend()
save_fig(FIG_DIR, "02_markup")
print(f"AUC rasio pengembalian saja (baris terisi): {roc_auc_score(y[markup.notna()], markup.dropna()):.4f} | AUC jumlah pinjaman saja: {roc_auc_score(y, train.jumlah_pinjaman):.4f}")

**Temuan.** Ini temuan utama. Pinjaman dengan rasio pengembalian sampai 1,01 **tidak satu pun** gagal bayar (8.871 transaksi), rasio 1,05–1,1 gagal bayar 16%, rasio 1,1–1,2 sebanyak 60%, dan di atas 1,2 sebanyak 92%. Rasio pengembalian sendirian memberi AUC 0,984, sementara jumlah pinjaman hanya 0,618. Kemungkinan besar total pengembalian sudah memuat denda atau bunga keterlambatan, sehingga nilainya baru diketahui setelah pinjaman berjalan. Untuk kompetisi atribut ini sah dipakai, tetapi untuk credit scoring sungguhan saat pengajuan, atribut ini tidak tersedia (lihat Keterbatasan).

### 3.3 Durasi, waktu pencairan, dan porsi lender

In [ ]:
t = pd.to_datetime(train.tanggal_pencairan)
q = train.groupby(t.dt.to_period("Q"))[TARGET].agg(["mean", "size"])
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
db = train.groupby(pd.cut(train.durasi_hari, [-2, 0, 6, 7, 14, 30, 60, 2000]), observed=True)[TARGET].agg(["mean", "size"])
ax[0].bar([str(i) for i in db.index], db["mean"], color=C1); pct_axis(ax[0], 0)
ax[0].set(xlabel="durasi (hari)", ylabel="proporsi gagal bayar", title="Gagal bayar per durasi"); ax[0].tick_params(axis="x", rotation=30)
ax2 = ax[1].twinx()
ax2.bar(q.index.astype(str), q["size"], color=CN, alpha=0.35)
ax[1].plot(q.index.astype(str), q["mean"], marker="o", color=C2); ax[1].set_zorder(ax2.get_zorder() + 1); ax[1].patch.set_visible(False)
pct_axis(ax[1], 0); ax[1].set(ylabel="proporsi gagal bayar", title="Per kuartal pencairan (batang = jumlah)"); ax[1].tick_params(axis="x", rotation=60)
ax2.set_ylabel("jumlah transaksi")
share = (train.porsi_pengembalian_lender / train.total_pengembalian.replace(0, np.nan)).round(2)
top = share.value_counts().head(8).index
sb = train[share.isin(top)].groupby(share[share.isin(top)])[TARGET].agg(["mean", "size"])
ax[2].bar(sb.index.astype(str), sb["mean"], color=C3); pct_axis(ax[2], 0)
ax[2].set(xlabel="porsi lender (8 nilai tersering)", ylabel="proporsi gagal bayar", title="Gagal bayar per porsi lender")
save_fig(FIG_DIR, "03_duration_time_share")
q.T.style.format("{:.3f}")

**Temuan.** Durasi 7 hari adalah standar (17.795 transaksi, 3,5% gagal bayar), dan risiko naik untuk durasi lebih panjang, sampai 57% untuk lebih dari 60 hari. Waktu pencairan juga berpengaruh: 2023Q3–Q4 memuat 91% transaksi dengan gagal bayar sekitar 3%, sedangkan kuartal lain gagal bayar 13–77%. Porsi lender 0,13–0,2 berisiko lebih tinggi karena nilai itu milik jenis pinjaman non-modal usaha, bukan karena porsinya sendiri.

### 3.4 Data kosong di train dan test tidak sama

In [ ]:
mt = train.total_pengembalian.isna()
print(f"Total pengembalian kosong: train {mt.mean():.1%}, test {test.total_pengembalian.isna().mean():.1%}")
print(f"Gagal bayar saat total pengembalian kosong: {y[mt].mean():.2%} vs terisi {y[~mt].mean():.2%}")
Xb_tr, Xb_te = base_frame(train, test)
adv_X = pd.concat([Xb_tr, Xb_te], ignore_index=True)
adv_y = np.r_[np.zeros(len(train)), np.ones(len(test))]
cv = StratifiedKFold(5, shuffle=True, random_state=0)
adv_p = cross_val_predict(lgb_model(0, n_estimators=200), adv_X, adv_y, cv=cv, method="predict_proba")[:, 1]
full = adv_X.notna().all(axis=1).values
adv_p2 = cross_val_predict(lgb_model(0, n_estimators=200), adv_X[full], adv_y[full], cv=cv, method="predict_proba")[:, 1]
print(f"Adversarial AUC train vs test: {roc_auc_score(adv_y, adv_p):.3f} | hanya baris tanpa nilai kosong: {roc_auc_score(adv_y[full], adv_p2):.3f}")
miss_tr = train[NUMERIC + ['jenis_pinjaman']].isna().any(axis=1)
print(f"Baris train dengan minimal satu nilai kosong: {miss_tr.mean():.1%} | test: {test[NUMERIC + ['jenis_pinjaman']].isna().any(axis=1).mean():.1%}")

**Temuan.** Saat total pengembalian kosong, gagal bayar 5,4%, hampir sama dengan saat terisi (5,0%). Jadi nilai kosong ini disisipkan acak dan tidak membawa informasi label. Adversarial AUC 0,607 seluruhnya berasal dari pola kosong: pada baris tanpa nilai kosong AUC turun ke 0,524. Konsekuensinya, skor cross-validation pada semua baris **meremehkan** skor test, karena di test rasio pengembalian hampir selalu bisa dihitung.

## 4. Preprocessing dan desain validasi

Keputusan yang diambil dari EDA:

- **Nilai kosong dibiarkan** untuk ditangani LightGBM, karena polanya tidak informatif terhadap label.
- **Kategori memakai satu kamus** untuk train dan test.
- **Validasi meniru test.** Total pengembalian kosong di 10% baris train tetapi hanya 1% baris test. Model tetap dilatih
  pada semua baris, tetapi skor utama dihitung pada **baris train yang total pengembaliannya terisi**, karena itulah kondisi
  hampir semua baris test. Skor pada semua baris juga dilaporkan.
- **Ambang dipilih dari OOF.** Macro F1 butuh label 0/1, jadi ambang probabilitas dipilih per seed dari prediksi
  out-of-fold, bukan ditetapkan 0,5.

In [ ]:
SHARE = typical_lender_share(train)
TEST_LIKE = train.total_pengembalian.notna().values
print("Porsi lender yang umum per jenis pinjaman:", SHARE)
print(f"Baris 'seperti test': {TEST_LIKE.sum():,} dari {len(train):,}")
RESULTS, OOF = {}, {}

def record(name, oof):
    f_tl, thr = macro_f1_by_seed(oof, y, TEST_LIKE)
    f_all, _ = macro_f1_by_seed(oof, y)
    o = np.atleast_2d(oof)
    OOF[name] = o.mean(0)
    RESULTS[name] = {"macro F1 (seperti test)": np.mean(f_tl), "std antar seed": np.std(f_tl), "macro F1 (semua baris)": np.mean(f_all),
                     "AUC": np.mean([roc_auc_score(y, p) for p in o]), "ambang": np.mean(thr)}
    return f"{name}: macro F1 seperti test {np.mean(f_tl):.4f} ± {np.std(f_tl):.4f} | semua baris {np.mean(f_all):.4f} | ambang {np.mean(thr):.2f}"

## 5. Baseline

Semua skor adalah hasil out-of-fold dari Stratified 5-Fold yang diulang dengan 5 seed.

- **Aturan satu atribut:** gagal bayar jika status peminjam = baru atau jenis pinjaman = konsumtif.
- **Regresi logistik:** atribut mentah, one-hot untuk kategori, median untuk nilai kosong, log untuk nominal uang.
- **LightGBM:** atribut mentah tanpa fitur turunan.

In [ ]:
rule = ((train.status_peminjam == "Peminjam_Baru") | (train.jenis_pinjaman == "Konsumtif")).astype(float).values
print(record("aturan: peminjam baru atau konsumtif", np.tile(rule, (len(SEEDS), 1))))
Xlr = train[NUMERIC + CATEGORICAL].copy()
for c in ["jumlah_pinjaman", "total_pengembalian", "porsi_pengembalian_lender"]:
    Xlr[c] = np.log1p(Xlr[c].clip(lower=0))
def logreg(seed):
    pre = make_column_transformer((make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
                                  (make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CATEGORICAL))
    return make_pipeline(pre, LogisticRegression(max_iter=3000, C=1.0))
print(record("regresi logistik, atribut mentah", cv_predict(logreg, Xlr, y, SEEDS)))
print(record("LightGBM, atribut mentah", cv_predict(lgb_model, Xb_tr, y, SEEDS)))

**Temuan.** Aturan "peminjam baru atau konsumtif" hanya mencapai macro F1 0,581. Regresi logistik pada atribut mentah 0,658, karena hubungan yang penting adalah rasio dua kolom, bukan kolom itu sendiri. LightGBM pada atribut mentah sudah 0,851: pohon bisa mendekati rasio dengan memotong total pengembalian dan jumlah pinjaman berulang kali, tetapi tidak sempurna.

## 6. Feature engineering dan ablation

Empat kelompok fitur turunan, masing-masing dari temuan EDA, ditambahkan satu per satu:

| Kelompok | Fitur | Dasar dari EDA |
| --- | --- | --- |
| A. rasio pengembalian | total / pokok, dan perkiraannya dari porsi lender jika total kosong | bagian 3.2 |
| B. porsi lender | porsi lender terhadap total dan terhadap pokok | bagian 3.3 |
| C. waktu pencairan | bulan ke-, hari dalam minggu | bagian 3.3 |
| D. pola data kosong | jumlah atribut kosong per baris | bagian 3.4 |

In [ ]:
ABL = []
steps = []
for g in GROUPS:
    steps.append(g)
    name = "LightGBM + " + ", ".join(s.split(".")[0] for s in steps)
    X = build(Xb_tr, train, SHARE, steps)
    print(record(name, cv_predict(lgb_model, X, y, SEEDS)))
    ABL.append((g, name, X.shape[1]))
abl = pd.DataFrame([{"langkah": "atribut mentah", "fitur": Xb_tr.shape[1], **RESULTS["LightGBM, atribut mentah"]}] +
                   [{"langkah": "+ " + g, "fitur": k, **RESULTS[n]} for g, n, k in ABL])
abl["perubahan"] = abl["macro F1 (seperti test)"].diff()
fig, ax = plt.subplots(figsize=(11, 4.3))
ax.errorbar(abl.langkah, abl["macro F1 (seperti test)"], yerr=abl["std antar seed"], marker="o", color=C2, capsize=4, label="baris seperti test")
ax.plot(abl.langkah, abl["macro F1 (semua baris)"], marker="s", color=C1, ls="--", label="semua baris")
ax.set(ylabel="macro F1 OOF", title="Ablation kumulatif kelompok fitur"); ax.legend(); ax.tick_params(axis="x", rotation=15)
save_fig(FIG_DIR, "04_ablation")
abl.style.format({c: "{:.4f}" for c in ["macro F1 (seperti test)", "std antar seed", "macro F1 (semua baris)", "AUC", "perubahan"]} | {"ambang": "{:.2f}"})

**Temuan.** Menambahkan rasio pengembalian (A) menaikkan macro F1 dari 0,851 menjadi **0,940**, lompatan terbesar di seluruh notebook. Porsi lender (B) tidak membantu (−0,0004). Waktu pencairan (C) menambah 0,007, lebih besar dari variasi antar seed. Pola data kosong (D) tidak menambah apa pun, sesuai temuan bahwa nilai kosong disisipkan acak. Pola yang sama terlihat pada skor semua baris, yang selalu sekitar 0,015 lebih rendah.

### 6.1 Set fitur akhir

Kelompok dipertahankan jika menaikkan macro F1 (seperti test) lebih dari satu standar deviasi antar seed dibanding langkah
sebelumnya; kelompok lain dibuang.

In [ ]:
KEEP = []
prev = RESULTS["LightGBM, atribut mentah"]["macro F1 (seperti test)"]
for g, n, _ in ABL:
    gain = RESULTS[n]["macro F1 (seperti test)"] - prev
    if gain > RESULTS[n]["std antar seed"]:
        KEEP.append(g)
    prev = RESULTS[n]["macro F1 (seperti test)"]
KEEP = KEEP or ["A. rasio pengembalian"]
FINAL_NAME = "LightGBM akhir: " + ", ".join(s.split(".")[0] for s in KEEP)
X_final = build(Xb_tr, train, SHARE, KEEP)
OOF_FINAL = cv_predict(lgb_model, X_final, y, SEEDS)
print("Kelompok dipertahankan:", KEEP)
print(record(FINAL_NAME, OOF_FINAL))

### 6.2 Tanpa total pengembalian: model saat pengajuan pinjaman

Total pengembalian dan porsi lender kemungkinan baru diketahui setelah pinjaman berjalan. Untuk melihat seberapa banyak
informasi tersisa saat pengajuan, model dilatih tanpa kedua kolom itu (tetap dengan waktu pencairan).

In [ ]:
X_pre = build(Xb_tr.drop(columns=["total_pengembalian", "porsi_pengembalian_lender"]), train, SHARE, ["C. waktu pencairan"])
print(record("LightGBM tanpa total pengembalian & porsi lender", cv_predict(lgb_model, X_pre, y, SEEDS)))

**Temuan.** Tanpa total pengembalian dan porsi lender, macro F1 turun dari 0,947 menjadi **0,667** (AUC 0,796), hanya sedikit di atas regresi logistik. Jadi hampir seluruh kemampuan model datang dari atribut yang kemungkinan baru diketahui setelah pinjaman berjalan. Untuk keputusan pemberian pinjaman, atribut yang tersedia saat pengajuan hanya cukup untuk penyaringan kasar.

## 7. Ambang keputusan untuk macro F1

In [ ]:
p_fin = OOF[FINAL_NAME]
curve_tl, curve_all = f1_curve(y[TEST_LIKE], p_fin[TEST_LIKE]), f1_curve(y, p_fin)
THR, F1_TL = best_threshold(y[TEST_LIKE], p_fin[TEST_LIKE])
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].plot(THRESHOLDS, curve_tl, color=C2, label="baris seperti test")
ax[0].plot(THRESHOLDS, curve_all, color=C1, ls="--", label="semua baris")
ax[0].axvline(THR, color="black", lw=0.8, ls=":", label=f"ambang terpilih {THR:.2f}")
ax[0].axvline(0.5, color=CN, lw=0.8, ls="--", label="0,5")
ax[0].set(xlabel="ambang probabilitas", ylabel="macro F1 OOF", title="Macro F1 terhadap ambang"); ax[0].legend(fontsize=8)
cm = confusion_matrix(y[TEST_LIKE], p_fin[TEST_LIKE] >= THR)
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax[1], xticklabels=["prediksi lancar", "prediksi gagal"], yticklabels=["lancar", "gagal bayar"])
ax[1].set(title=f"Confusion matrix OOF (seperti test), ambang {THR:.2f}")
save_fig(FIG_DIR, "05_threshold")
print(f"Macro F1 pada ambang 0,5: {f1_score(y[TEST_LIKE], p_fin[TEST_LIKE] >= 0.5, average='macro'):.4f} | pada ambang {THR:.2f}: {F1_TL:.4f}")
print(f"F1 kelas gagal bayar: {f1_score(y[TEST_LIKE], p_fin[TEST_LIKE] >= THR):.4f} | F1 kelas lancar: {f1_score(1 - y[TEST_LIKE], p_fin[TEST_LIKE] < THR):.4f}")

**Temuan.** Kurva macro F1 cukup datar di sekitar ambang 0,3–0,55, jadi pilihan ambang tidak sensitif. Ambang terpilih 0,42 memberi 0,9487 pada OOF gabungan, hanya sedikit di atas ambang 0,5 (0,9476). F1 kelas gagal bayar 0,903 dan kelas lancar 0,995; macro F1 terutama ditentukan oleh kelas gagal bayar.

## 8. Interpretasi

### 8.1 Fitur terpenting

In [ ]:
m = lgb_model(0).fit(X_final, y)
imp = pd.Series(m.booster_.feature_importance("gain"), index=X_final.columns)
imp = (imp / imp.sum()).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(16, 4.8))
ax[0].barh(imp.index, imp.values, color=[C2 if c in sum(GROUPS.values(), []) else C1 for c in imp.index])
pct_axis(ax[0], 0, axis="x"); ax[0].set(xlabel="porsi total gain", title="Kepentingan fitur (oranye = turunan)")
der = derived(train, SHARE)
grid = np.linspace(1.0, 1.4, 41)
probe = X_final[TEST_LIKE].sample(2000, random_state=0).copy()
pdp = []
for g in grid:
    probe["markup"] = g
    pdp.append(m.predict_proba(probe)[:, 1].mean())
ax[1].plot(grid, pdp, color=C2)
ax[1].axhline(THR, color=CN, ls=":", lw=1, label=f"ambang {THR:.2f}"); ax[1].legend()
ax[1].set(xlabel="rasio pengembalian (diset untuk semua baris)", ylabel="rata-rata peluang gagal bayar", title="Partial dependence rasio pengembalian")
save_fig(FIG_DIR, "06_importance")

**Temuan.** Rasio pengembalian menyumbang porsi gain terbesar, disusul bulan pencairan dan jenis pinjaman. Partial dependence menunjukkan peluang gagal bayar hampir nol sampai rasio sekitar 1,05, lalu naik tajam dan melewati ambang keputusan di kisaran rasio 1,1.

### 8.2 Di mana model salah?

In [ ]:
pred = (p_fin >= THR).astype(int)
err = pd.DataFrame({"label": y, "pred": pred, "markup": der.markup, "status": train.status_peminjam, "jenis": train.jenis_pinjaman,
                    "total kosong": ~TEST_LIKE})
err["jenis kesalahan"] = np.select([(err.label == 1) & (err.pred == 0), (err.label == 0) & (err.pred == 1)], ["gagal bayar terlewat", "alarm palsu"], "benar")
tab = err[err["jenis kesalahan"] != "benar"].groupby("jenis kesalahan").agg(jumlah=("label", "size"), median_rasio=("markup", "median"),
        porsi_total_kosong=("total kosong", "mean"), porsi_peminjam_baru=("status", lambda s: (s == "Peminjam_Baru").mean()))
fig, ax = plt.subplots(figsize=(10, 4.3))
for k, col, lab in [("benar", CN, "benar"), ("gagal bayar terlewat", C2, "gagal bayar terlewat"), ("alarm palsu", C1, "alarm palsu")]:
    s = err.markup[err["jenis kesalahan"] == k].dropna().clip(0.95, 1.6)
    ax.hist(s, bins=60, alpha=0.6, density=True, color=col, label=f"{lab} (n={len(s):,})")
ax.set(yscale="log", xlabel="rasio pengembalian", ylabel="densitas (log)", title="Rasio pengembalian menurut hasil prediksi"); ax.legend()
save_fig(FIG_DIR, "07_errors")
tab.style.format({"median_rasio": "{:.3f}", "porsi_total_kosong": "{:.0%}", "porsi_peminjam_baru": "{:.0%}"})

**Temuan.** Kesalahan terbagi hampir seimbang: 133 gagal bayar terlewat dan 116 alarm palsu. **41% gagal bayar yang terlewat adalah baris yang total pengembaliannya kosong**, sehingga rasio tidak bisa dihitung. Gagal bayar terlewat lainnya punya rasio rendah (median 1,061), sedangkan alarm palsu adalah pinjaman lancar dengan rasio tinggi (median 1,131). Keduanya ada di wilayah tumpang tindih rasio 1,05–1,2.

## 9. Submission

Model akhir dilatih ulang pada seluruh data latih dengan 5 seed, lalu probabilitas rata-ratanya diubah menjadi label dengan ambang dari bagian 7.

In [ ]:
X_test = build(Xb_te, test, SHARE, KEEP)
p_test = np.mean([lgb_model(s).fit(X_final, y).predict_proba(X_test)[:, 1] for s in SEEDS], axis=0)
sub = sample[[ID]].merge(pd.DataFrame({ID: test[ID], TARGET: (p_test >= THR).astype(int)}), on=ID, how="left")
assert sub[TARGET].notna().all() and len(sub) == len(sample)
sub.to_csv(OUT_DIR / "submission.csv", index=False)
res = pd.DataFrame(RESULTS).T.sort_values("macro F1 (seperti test)")
res.to_csv(OUT_DIR / "cv_results.csv"); abl.to_csv(OUT_DIR / "ablation.csv", index=False)
print(f"Ambang {THR:.2f} | prediksi gagal bayar di test: {sub[TARGET].mean():.2%} (train {y.mean():.2%})")
res.style.format({c: "{:.4f}" for c in res.columns if c != "ambang"} | {"ambang": "{:.2f}"})

**Temuan.** Proporsi prediksi gagal bayar di test 5,3%, dekat dengan 5,03% di train, sehingga ambang dari OOF terlihat cocok untuk test.

## 10. Kesimpulan

1. **Satu rasio menjelaskan hampir semuanya.** Total pengembalian dibagi pokok menaikkan macro F1 dari 0,851 ke 0,940; atribut profil peminjam hampir tidak berperan.
2. **Validasi harus meniru test.** Nilai kosong 10 kali lebih banyak di train, sehingga skor dihitung pada baris yang lengkap. Tanpa itu, skor CV meremehkan skor test sekitar 0,015.
3. **Hasil.** Macro F1 OOF 0,947, late submission private 0,9450, sedikit di bawah submission tim terbaik saat kompetisi (0,9471) dan 0,016 di bawah peringkat 1 (0,9614).

**Keterbatasan.** Total pengembalian kemungkinan dicatat setelah pinjaman berjalan, sehingga model ini lebih tepat disebut deteksi gagal bayar daripada credit scoring saat pengajuan. Tanpa atribut itu, macro F1 turun ke 0,667 (bagian 6.2), sehingga model untuk keputusan pemberian pinjaman butuh data peminjam yang lebih kaya.

### Jejak lingkungan

In [ ]:
import platform
pkgs = ["numpy", "pandas", "scikit-learn", "lightgbm", "matplotlib", "seaborn"]
print(" | ".join([f"python {platform.python_version()}"] + [f"{p} {version(p)}" for p in pkgs]))
print(f"Waktu eksekusi total: {(time.time() - T0) / 60:.1f} menit")